In [1]:
import sqlite3
print("SQLite is available")
print(sqlite3.sqlite_version)

SQLite is available
3.45.1


In [2]:
import sqlalchemy
print(sqlalchemy.__version__)

2.0.52


In [3]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory")
for name in ['sales', 'products', 'customers']:
    pd.read_csv(f"{name}.csv").to_sql(name, conn, index=False, if_exists='replace')

def run(sql):
    return pd.read_sql_query(sql, conn)

print("Database ready with 3 tables: sales, products, customers")

Database ready with 3 tables: sales, products, customers


Part A — 20 SQL queries (the deliverable)

TASK 01 

Twenty queries of increasing difficulty

The graded deliverable. Solve all 20, each with a plain-English comment above it

• Easy (SELECT / WHERE):

In [5]:
# Show the first 10 rows of the sales table.
run("SELECT * FROM sales LIMIT 10")

,order_id,date,customer_id,product_id,quantity
0,O00001,2024-10-31,C999,P019,2
1,O00002,2024-03-14,C999,P014,1
2,O00003,2024-10-21,C999,P004,1
3,O00004,2024-01-03,C999,P003,3
4,O00005,2024-10-18,C999,P018,4
5,O00006,2024-10-15,C010,P006,2
6,O00007,2024-10-12,C013,P016,5
7,O00008,2024-08-31,C046,P003,2
8,O00009,2024-06-21,C048,P006,5
9,O00010,2024-09-15,C003,P006,2


In [6]:
# Count the total number of orders.
run("SELECT COUNT(*) FROM sales")

,COUNT(*)
0,1000


In [7]:
# Show all orders with a quantity of 5.
run("SELECT * FROM sales WHERE quantity = 5")

,order_id,date,customer_id,product_id,quantity
0,O00007,2024-10-12,C013,P016,5
1,O00009,2024-06-21,C048,P006,5
2,O00014,2024-06-16,C015,P019,5
3,O00020,2024-02-11,C025,P015,5
4,O00022,2024-09-01,C005,P020,5
...,...,...,...,...,...
186,O00981,2024-12-02,C042,P009,5
187,O00988,2024-12-28,C034,P004,5
188,O00989,2024-08-26,C016,P005,5
189,O00990,2024-02-21,C001,P010,5


In [8]:
# List all products in the 'Electronics' category.
run("SELECT * FROM products WHERE category = 'Electronics'")

,product_id,product_name,category,unit_price
0,P001,Laptop,Electronics,800
1,P002,Mouse,Electronics,25
2,P003,Keyboard,Electronics,45
3,P004,Monitor,Electronics,250
4,P005,Webcam,Electronics,60
5,P006,Headphones,Electronics,120
6,P007,USB Hub,Electronics,35
7,P016,Cable Set,Electronics,30
8,P017,Power Bank,Electronics,50
9,P018,Speaker,Electronics,90


In [9]:
# Show the 5 most expensive products (by unit price).
run("SELECT * FROM products ORDER BY unit_price DESC LIMIT 5")

,product_id,product_name,category,unit_price
0,P001,Laptop,Electronics,800
1,P010,Standing Desk,Office,400
2,P020,Tablet,Electronics,350
3,P004,Monitor,Electronics,250
4,P009,Chair,Office,180


• Medium (GROUP BY / aggregates / HAVING):

In [10]:
# Count how many customers are in each region.
run('''
    SELECT region, COUNT(*) AS customer_id
    FROM customers
    GROUP BY region
    ORDER BY customer_id DESC
''')

,region,customer_id
0,West,16
1,East,13
2,North,12
3,South,11


In [11]:
# Find the total quantity sold for each product.
run('''
    SELECT product_id, SUM(quantity) AS total_quantity
    FROM sales
    GROUP BY product_id
''')

,product_id,total_quantity
0,P001,126
1,P002,138
2,P003,159
3,P004,177
4,P005,171
5,P006,145
6,P007,159
7,P008,178
8,P009,156
9,P010,172


In [12]:
# Find the average unit price per category.
run('''
   SELECT category, AVG(unit_price) AS avg_unit_price
   FROM products
   GROUP BY category
''')

,category,avg_unit_price
0,Accessories,31.000000
1,Electronics,163.750000
2,Office,206.666667
3,Stationery,11.500000


In [13]:
# Count orders per product, but only show products with more than 50 orders (use HAVING).
run('''
    SELECT product_id, COUNT(*) AS orders
    FROM sales
    GROUP BY product_id
    HAVING COUNT(*) > 50
    ORDER BY orders DESC
''')

,product_id,orders
0,P004,62
1,P008,59
2,P019,57
3,P003,56
4,P010,54
5,P005,54
6,P020,53
7,P007,53
8,P017,51


In [14]:
# Find the single highest and lowest unit price in the products table.
run('''
   SELECT 
   MIN(unit_price) AS Lowest,
   MAX(unit_price) AS Highest
   FROM products
''')

,Lowest,Highest
0,8,800


• Harder (JOINs):

In [15]:
# Join sales to products and show each order's product name and price.
run('''
   SELECT s.order_id, s.product_id, p.product_name, p.unit_price
   FROM sales s
   JOIN products p 
   ON s.product_id = p.product_id
''')

,order_id,product_id,product_name,unit_price
0,O00001,P019,Microphone,110
1,O00002,P014,Water Bottle,20
2,O00003,P004,Monitor,250
3,O00004,P003,Keyboard,45
4,O00005,P018,Speaker,90
...,...,...,...,...
995,O00996,P016,Cable Set,30
996,O00997,P004,Monitor,250
997,O00998,P007,USB Hub,35
998,O00999,P004,Monitor,250


In [16]:
# Compute total revenue (quantity times price) across all sales.
run('''
    SELECT
    SUM(quantity * unit_price) AS total_revenue
    FROM sales s
    JOIN products p
    ON s.product_id = p.product_id
''')

,total_revenue
0,405150


In [17]:
# Compute total revenue per region (join all three tables).
run('''
    SELECT c.region,
    SUM(s.quantity * p.unit_price) AS total_revenue
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    JOIN products p ON s.product_id = p.product_id
    GROUP BY c.region
    ORDER BY total_revenue DESC
''')

,region,total_revenue
0,East,121636
1,West,117271
2,North,83444
3,South,81814


In [18]:
# Compute total revenue per product category.
run('''
   SELECT p.category,
   SUM(s.quantity * p.unit_price) AS total_revenue
   FROM sales s
   JOIN products p ON s.product_id = p.product_id
   GROUP BY p.category
   ORDER BY total_revenue DESC
''')

,category,total_revenue
0,Electronics,285690
1,Office,104000
2,Accessories,12081
3,Stationery,3379


In [19]:
# Find how many sales reference a customer 
# who isn't in the customers table (hint: LEFT JOIN + IS NULL).
run('''
    SELECT COUNT(*) AS missing_customer
    FROM sales s
    LEFT JOIN customers c ON s.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
''')

,missing_customer
0,5


• Advanced (windows / CTEs / subqueries):

In [20]:
# Rank all products by total revenue using a window function.
run('''
    SELECT product_id, revenue, 
    Rank() OVER(ORDER BY revenue DESC) AS revenue_rank
    FROM(
        SELECT s.product_id, SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s 
        JOIN products p ON s.product_id = p.product_id
        GROUP BY s.product_id
    )
    ORDER BY revenue_rank
''')

,product_id,revenue,revenue_rank
0,P001,100800,1
1,P010,68800,2
2,P020,54600,3
3,P004,44250,4
4,P009,28080,5
5,P019,18590,6
6,P006,17400,7
7,P018,11520,8
8,P005,10260,9
9,P017,8050,10


In [21]:
# Compute a running total of revenue by month.
run('''
    WITH monthly AS(
        SELECT substr(date, 1, 7) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s 
        JOIN products p ON s.product_id = p.product_id
        GROUP BY substr(date, 1, 7)
    )
    SELECT month, revenue,
    SUM(revenue) OVER (ORDER BY month) AS running_total
    FROM monthly
''')

,month,revenue,running_total
0,2024-01,32561,32561
1,2024-02,47995,80556
2,2024-03,38608,119164
3,2024-04,28702,147866
4,2024-05,27291,175157
5,2024-06,31720,206877
6,2024-07,34522,241399
7,2024-08,31247,272646
8,2024-09,29860,302506
9,2024-10,30620,333126


In [22]:
# Find the top-selling product within each category (PARTITION BY).
run('''
    SELECT category, product_name, revenue 
    FROM(
        SELECT p.category, p.product_name,
        SUM(s.quantity * p.unit_price) AS revenue,
        RANK() OVER (PARTITION BY p.category
                        ORDER BY SUM(s.quantity * p.unit_price) DESC) AS rank
        FROM sales s 
        JOIN products p ON s.product_id = p.product_id
        GROUP BY p.product_id
    )
    WHERE rank = 1
''')

,category,product_name,revenue
0,Accessories,Backpack,7315
1,Electronics,Laptop,100800
2,Office,Standing Desk,68800
3,Stationery,Pen Set,2115


In [23]:
# Using a CTE, find all products that sold above the average product revenue.
run('''
    WITH product_revenue AS (
        SELECT s.product_id, SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s 
        JOIN products p ON s.product_id = p.product_id
        GROUP BY s.product_id
    )
    SELECT product_id, revenue
    FROM product_revenue
    WHERE revenue > (SELECT AVG(revenue) FROM product_revenue)
    ORDER BY revenue DESC
''')

,product_id,revenue
0,P001,100800
1,P010,68800
2,P020,54600
3,P004,44250
4,P009,28080


In [24]:
# Compute each region's revenue as a percentage of total revenue.
run('''
    WITH region_revenue AS(
        SELECT c.region, SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s 
        JOIN customers c ON s.customer_id = c.customer_id
        JOIN products p ON s.product_id = p.product_id
        GROUP BY c.region
    )
    SELECT region, revenue,
    ROUND(revenue * 100 / (SELECT SUM(revenue) FROM region_revenue), 2)
    AS revenue_percentage
    FROM region_revenue
''')

,region,revenue,revenue_percentage
0,East,121636,30.0
1,North,83444,20.0
2,South,81814,20.0
3,West,117271,29.0


Part B — the same analysis in Pandas AND SQL (the deliverable)

TASK 02 

Side-by-side: Pandas vs SQL

The second deliverable. Solve one real analysis both ways and compare.

In [25]:
# Compute total revenue per region per month.
import pandas as pd

sales = pd.read_csv('sales.csv')
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')

# Pandas Version

full = (
    sales
    .merge(customers, on="customer_id", how='left')
    .merge(products, on="product_id", how='left')
)

full['revenue'] = full['quantity'] * full['unit_price']

full['date'] = pd.to_datetime(full['date'])

full['month'] = full['date'].dt.to_period("M")

result = (full.groupby(['region', 'month'])['revenue']
         .sum().reset_index())

print("\n" + "=" * 20)
print("Pandas Result")
print("=" * 20)
print(result)


Pandas Result
   region    month  revenue
0    East  2024-01    11506
1    East  2024-02    11110
2    East  2024-03    13911
3    East  2024-04    10275
4    East  2024-05     5976
5    East  2024-06     6378
6    East  2024-07     9568
7    East  2024-08    12035
8    East  2024-09    12640
9    East  2024-10     8635
10   East  2024-11     5029
11   East  2024-12    14573
12  North  2024-01     2975
13  North  2024-02    12119
14  North  2024-03     7269
15  North  2024-04     8534
16  North  2024-05    11601
17  North  2024-06     6915
18  North  2024-07     5573
19  North  2024-08     3604
20  North  2024-09     4852
21  North  2024-10     6562
22  North  2024-11     5831
23  North  2024-12     7609
24  South  2024-01     6114
25  South  2024-02    12525
26  South  2024-03    10415
27  South  2024-04     3930
28  South  2024-05     6320
29  South  2024-06     3685
30  South  2024-07     5078
31  South  2024-08     6931
32  South  2024-09     7848
33  South  2024-10     6331
34  S

In [26]:
# SQL Version
print("\n" + "=" * 20)
print("SQL Result")
print("=" * 20)

sql_result = pd.read_sql_query('''
    SELECT c.region,
    strftime('%Y-%m', s.date) AS month,
    SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
    LEFT JOIN customers c ON s.customer_id = c.customer_id
    LEFT JOIN products p ON s.product_id = p.product_id
    GROUP BY c.region,
    strftime('%Y-%m', s.date)
    ORDER BY c.region, month
''', conn)

print(sql_result)


SQL Result
   region    month  revenue
0     NaN  2024-01      135
1     NaN  2024-03       20
2     NaN  2024-10      830
3    East  2024-01    11506
4    East  2024-02    11110
5    East  2024-03    13911
6    East  2024-04    10275
7    East  2024-05     5976
8    East  2024-06     6378
9    East  2024-07     9568
10   East  2024-08    12035
11   East  2024-09    12640
12   East  2024-10     8635
13   East  2024-11     5029
14   East  2024-12    14573
15  North  2024-01     2975
16  North  2024-02    12119
17  North  2024-03     7269
18  North  2024-04     8534
19  North  2024-05    11601
20  North  2024-06     6915
21  North  2024-07     5573
22  North  2024-08     3604
23  North  2024-09     4852
24  North  2024-10     6562
25  North  2024-11     5831
26  North  2024-12     7609
27  South  2024-01     6114
28  South  2024-02    12525
29  South  2024-03    10415
30  South  2024-04     3930
31  South  2024-05     6320
32  South  2024-06     3685
33  South  2024-07     5078
34  Sout

In [27]:
# Compare the two results
pandas_check = result.copy()
pandas_check['month'] = pandas_check['month'].astype(str)
pandas_check = (pandas_check.sort_values(
    ['region', 'month']
).reset_index(drop=True))

pandas_check['revenue'] = pandas_check['revenue'].round(2)

sql_check = sql_result.copy()
sql_check = (sql_check.sort_values(['region', 'month']).reset_index(drop=True))
sql_check['revenue'] = sql_result['revenue'].round(2)

print("Pandas Result:")
print(pandas_check)
print("\nSQL Result:")
print(sql_check)
print("\nDo results match?")
print(pandas_check.equals(sql_check))

Pandas Result:
   region    month  revenue
0    East  2024-01    11506
1    East  2024-02    11110
2    East  2024-03    13911
3    East  2024-04    10275
4    East  2024-05     5976
5    East  2024-06     6378
6    East  2024-07     9568
7    East  2024-08    12035
8    East  2024-09    12640
9    East  2024-10     8635
10   East  2024-11     5029
11   East  2024-12    14573
12  North  2024-01     2975
13  North  2024-02    12119
14  North  2024-03     7269
15  North  2024-04     8534
16  North  2024-05    11601
17  North  2024-06     6915
18  North  2024-07     5573
19  North  2024-08     3604
20  North  2024-09     4852
21  North  2024-10     6562
22  North  2024-11     5831
23  North  2024-12     7609
24  South  2024-01     6114
25  South  2024-02    12525
26  South  2024-03    10415
27  South  2024-04     3930
28  South  2024-05     6320
29  South  2024-06     3685
30  South  2024-07     5078
31  South  2024-08     6931
32  South  2024-09     7848
33  South  2024-10     6331
34  S

Pandas vs SQL

For this particular question, Pandas is probably easier to write because merge(), groupby(), and .dt are straightforward. SQL can be easier to read for database-style analysis, because the SELECT → JOIN → GROUP BY structure clearly describes what data is being requested. I would choose Pandas when the data is already in Python and I need further data cleaning, analysis, or visualization. I would choose SQL when the data is stored in a database and I need to filter, join, aggregate, or retrieve large amounts of data efficiently before bringing the result into Python.

Part C — get comfortable with window functions

TASK 03 Window 

functions practice

The curriculum's window-function requirement (running total + rank).

In [28]:
# Running total of revenue over the whole year
run('''
    WITH daily_revenue AS (
        SELECT date, SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN products p ON s.product_id = p.product_id
        GROUP BY date
    )
    SELECT date, revenue, SUM(revenue) OVER (
            ORDER BY date
    ) AS running_total
    FROM daily_revenue
    ORDER BY date
''')

,date,revenue,running_total
0,2024-01-02,2654,2654
1,2024-01-03,3103,5757
2,2024-01-04,85,5842
3,2024-01-05,250,6092
4,2024-01-06,3745,9837
...,...,...,...
338,2024-12-27,1600,396001
339,2024-12-28,1625,397626
340,2024-12-29,280,397906
341,2024-12-30,3154,401060


In [31]:
# Rank customers by total spending Show top 10 customers
run('''
    WITH customer_spending AS (
        SELECT customer_id,
        SUM(s.quantity * p.unit_price) AS spend
        FROM sales s
        JOIN products p ON s.product_id = p.product_id
        GROUP BY customer_id
    ),
    ranked_customers AS (
        SELECT customer_id, spend, 
        RANK() OVER (
            ORDER BY spend DESC
        ) AS spending_rank
        FROM customer_spending
    )
    SELECT customer_id, spend, spending_rank
    FROM ranked_customers
    WHERE spending_rank <= 10
    ORDER BY spending_rank
''')

,customer_id,spend,spending_rank
0,C024,19300,1
1,C015,17480,2
2,C003,12885,3
3,C039,12568,4
4,C023,12464,5
5,C043,12214,6
6,C027,11775,7
7,C050,11644,8
8,C007,11114,9
9,C029,10887,10


In [34]:
# Rank customers within each region Ranking restarts for every region
run('''
    WITH customer_spending AS (
        SELECT c.region, s.customer_id,
        SUM(s.quantity * p.unit_price) AS spend
        FROM sales s
        JOIN products p ON s.product_id = p.product_id
        JOIN customers c ON s.customer_id = c.customer_id
        GROUP BY c.region, s.customer_id
    )
    SELECT region, customer_id, spend,
    RANK() OVER(
        PARTITION BY region
        ORDER BY spend DESC
    ) AS region_rank
    FROM customer_spending
    ORDER BY region, region_rank
''')

,region,customer_id,spend,region_rank
0,East,C015,17480,1
1,East,C003,12885,2
2,East,C043,12214,3
3,East,C029,10887,4
4,East,C034,10843,5
5,East,C049,9920,6
6,East,C021,9745,7
7,East,C008,8986,8
8,East,C017,5967,9
9,East,C011,5905,10


In [37]:
# Monthly revenue + previous month's revenue
run('''
    WITH monthly_revenue AS (
        SELECT strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN products p ON s.product_id = p.product_id
        GROUP BY strftime('%Y-%m', s.date)
    )
    SELECT month, revenue,
    LAG(revenue) OVER(ORDER BY month) AS prev_month_revenue
    FROM monthly_revenue
    ORDER BY month
''')

,month,revenue,prev_month_revenue
0,2024-01,32561,NaN
1,2024-02,47995,32561.0
2,2024-03,38608,47995.0
3,2024-04,28702,38608.0
4,2024-05,27291,28702.0
5,2024-06,31720,27291.0
6,2024-07,34522,31720.0
7,2024-08,31247,34522.0
8,2024-09,29860,31247.0
9,2024-10,30620,29860.0


In [40]:
# Month-over-month revenue growth
run('''
    WITH monthly_revenue AS (
        SELECT strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN products p ON s.product_id = p.product_id
        GROUP BY strftime('%Y-%m', s.date)
    ),
    monthly_with_previous AS(
        SELECT month, revenue, 
        LAG(revenue) OVER(ORDER BY month) AS prev_month_revenue
        FROM monthly_revenue
    )
    SELECT month, revenue, prev_month_revenue,
    ROUND(
        (revenue - prev_month_revenue) * 100 / NULLIF(prev_month_revenue, 0), 2
    ) AS mom_growth_percent
    FROM monthly_with_previous
    ORDER BY month
''')

,month,revenue,prev_month_revenue,mom_growth_percent
0,2024-01,32561,NaN,NaN
1,2024-02,47995,32561.0,47.0
2,2024-03,38608,47995.0,-19.0
3,2024-04,28702,38608.0,-25.0
4,2024-05,27291,28702.0,-4.0
5,2024-06,31720,27291.0,16.0
6,2024-07,34522,31720.0,8.0
7,2024-08,31247,34522.0,-9.0
8,2024-09,29860,31247.0,-4.0
9,2024-10,30620,29860.0,2.0
